<a href="https://colab.research.google.com/github/GilliardMorandim/mba-tcc-usp-inadimplencia/blob/eda%2Ffeature/pre_processing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

SEMANA 2 – 9 a 15 de Dezembro

Objetivo: Pré-processamento completo
Tarefas:

Normalização (z-score)

Tratamento de missings (MICE/KNN)

One-hot encoding / target encoding

Preparar dataset final para modelagem

Entrega:
Dataset “master” pré-processado.

- incluir a pct_do_contrato no parcelas_main para saber se é uma parcela do começo do contrato ou não
-

# Requirements

In [1]:
!apt-get install openjdk-11-jdk-headless -qq > /dev/null
!pip install pyspark


In [2]:
import matplotlib.pyplot as plt
from google.colab import drive
import os
import pandas as pd

drive.mount('/content/drive')


Mounted at /content/drive


# Lendo Arquivo Pyspark

In [3]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
         .appName("LoadAllCSVs")
         .config("spark.driver.memory", "8g")
         .config("spark.executor.memory", "8g")
         .config("spark.sql.files.maxPartitionBytes", "256m")
         .getOrCreate())

print("Spark iniciado!")

Spark iniciado!


In [4]:
dfs_spark = {}

base_path = "/content/drive/MyDrive/MBA - Ciencia de Dados - USP/dados_tcc/"

files = [f for f in os.listdir(base_path) if f.endswith(".csv")]

for file in files:
    full_path = os.path.join(base_path, file)
    print(f"\n📥 Lendo via PySpark: {file}")

    try:
        df = (spark.read
              .option("header", "true")
              .option("inferSchema", "true")
              .csv(full_path))

        key = file.replace(".csv", "")
        dfs_spark[key] = df

        print(f"✔ OK - Linhas (estimado pela Spark): {df.count()} | Colunas: {len(df.columns)}")

    except Exception as e:
        print(f"❌ Erro ao ler {file}: {e}")

print("\nTodos os arquivos foram processados!")

globals().update(dfs_spark)


📥 Lendo via PySpark: pre_aprovado.csv
✔ OK - Linhas (estimado pela Spark): 42171363 | Colunas: 11

📥 Lendo via PySpark: parcelas.csv
✔ OK - Linhas (estimado pela Spark): 1865444 | Colunas: 21

📥 Lendo via PySpark: contratos.csv
✔ OK - Linhas (estimado pela Spark): 382539 | Colunas: 26

📥 Lendo via PySpark: score_credito.csv
✔ OK - Linhas (estimado pela Spark): 266126 | Colunas: 3

📥 Lendo via PySpark: analise_conversao.csv
✔ OK - Linhas (estimado pela Spark): 538023 | Colunas: 12

📥 Lendo via PySpark: analise_conversao_v2.csv
✔ OK - Linhas (estimado pela Spark): 538023 | Colunas: 12

📥 Lendo via PySpark: analise_conversao_v3.csv
✔ OK - Linhas (estimado pela Spark): 533525 | Colunas: 11

Todos os arquivos foram processados!


# Pre-Processing

In [5]:
parcelas_main = parcelas

contratos_keys = contratos.select(
    "id_contrato",
    "uuid_cliente",
    "id_contrato_original",
    "id_contrato_pai",
    "cpf_hash_sha256").filter("id_contrato IS NOT NULL")


parcelas_main = parcelas.join(
    contratos_keys,
    on="id_contrato",
    how="left"
)

parcelas_main = parcelas_main.filter("data_vencimento < '2025-11-01'")

# Normalização da parcela

In [6]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# janela por contrato
w = Window.partitionBy("id_contrato")

parcelas_main = (
    parcelas_main

    .withColumn("min_parcela", F.min("parcela").over(w))
    .withColumn("max_parcela", F.max("parcela").over(w))
    .withColumn(
        "parcela_norm_0_1",
        F.round((F.col("parcela") - F.col("min_parcela")) /
        (F.col("max_parcela") - F.col("min_parcela")),2
    ))
    .drop("min_parcela", "max_parcela")
)

parcelas_main.orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)
#parcelas_main.show(10,truncate=False)

+--------------------------------------+--------------+---------------+------+---------+------------------------------+------------+-------------+------------+-----------+-----------+-----------------+----------+--------------+------------------+--------------------------+----------------+--------------------------------------+-------+------------------+-------------------------------+--------------------------------------+--------------------------------------+---------------+----------------------------------------------------------------+----------------+
|id_contrato                           |data_pagamento|data_vencimento|valor |valor_iof|valor_financiado_principal_iof|qtd_parcelas|valor_parcela|valor_tarifa|valor_juros|valor_iof_2|valor_amortizacao|valor_pago|valor_desconto|valor_juros_atraso|valor_juros_remuneratorios|status_pagamento|id_parcela                            |parcela|qtd_dias_de_atraso|contract_installments_status_id|uuid_cliente                          |id_contr

In [7]:
from pyspark.sql import functions as F
from pyspark.sql.window  import Window

#define a janela de particionamento de underbound
w_ffill =(Window.partitionBy("id_contrato")
          .orderBy("parcela")
          .rowsBetween(Window.unboundedPreceding, Window.currentRow))

parcelas_main = parcelas_main.withColumn("data_pagamento_aux",F.last("data_pagamento",ignorenulls=True).over(w_ffill))

parcelas_main = parcelas_main.withColumn("data_vencimento_aux",F.greatest
 (F.datediff(F.col("data_pagamento_aux"),F.col("data_vencimento")), F.lit(0)))
parcelas_main = parcelas_main.drop("data_vencimento_aux")

In [8]:
parcelas_main.orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)

+--------------------------------------+--------------+---------------+------+---------+------------------------------+------------+-------------+------------+-----------+-----------+-----------------+----------+--------------+------------------+--------------------------+----------------+--------------------------------------+-------+------------------+-------------------------------+--------------------------------------+--------------------------------------+---------------+----------------------------------------------------------------+----------------+------------------+
|id_contrato                           |data_pagamento|data_vencimento|valor |valor_iof|valor_financiado_principal_iof|qtd_parcelas|valor_parcela|valor_tarifa|valor_juros|valor_iof_2|valor_amortizacao|valor_pago|valor_desconto|valor_juros_atraso|valor_juros_remuneratorios|status_pagamento|id_parcela                            |parcela|qtd_dias_de_atraso|contract_installments_status_id|uuid_cliente                

# Flag Contrato sem nenhum pagamento

In [13]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

w_contrato = Window.partitionBy("id_contrato")

parcelas_main = parcelas_main.withColumn(
    "flag_contrato_sem_pagamento",
    F.when(
        F.max(F.col("data_pagamento").isNotNull().cast("int")).over(w_contrato) == 0,
        F.lit(1)
    ).otherwise(F.lit(0))
)


In [25]:
#Validando flag-inadimplencia over todas parcelas
parcelas_main.select(
    "id_contrato",
    "parcela",
    "data_vencimento",
    "data_pagamento",
    "data_pagamento_aux",
    "flag_contrato_sem_pagamento",
).orderBy("data_vencimento").filter(F.col("id_contrato") == "{00012C6D-DD36-401A-8B26-F0C46C121114}").show(100, truncate=False)

+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+
|id_contrato                           |parcela|data_vencimento|data_pagamento|data_pagamento_aux|flag_contrato_sem_pagamento|
+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+
|{00012C6D-DD36-401A-8B26-F0C46C121114}|1      |2025-07-25     |NULL          |NULL              |1                          |
|{00012C6D-DD36-401A-8B26-F0C46C121114}|2      |2025-08-25     |NULL          |NULL              |1                          |
|{00012C6D-DD36-401A-8B26-F0C46C121114}|3      |2025-09-25     |NULL          |NULL              |1                          |
+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+



# Data Pagamento Aux

In [22]:
w_primeira = (
    Window
    .partitionBy("id_contrato")
    .orderBy("parcela")
    .rowsBetween(Window.unboundedPreceding, Window.unboundedFollowing)
)

# Criação da coluna com a data de vencimento da primeira parcela
parcelas_main = parcelas_main.withColumn(
    "data_vencimento_primeira_parcela",
    F.first("data_vencimento", ignorenulls=True).over(w_primeira)
)


In [23]:
parcelas_main = parcelas_main.withColumn(
    "qtd_dias_de_atraso_v2",
    F.when(
        # 🔴 Cenário B — contrato nunca teve pagamento
        F.col("flag_contrato_sem_pagamento") == 1,
        F.greatest(
            F.datediff(
                F.col("data_vencimento"),
                F.col("data_vencimento_primeira_parcela")
            ),
            F.lit(0)
        )
    ).when(
        # 🟢 Cenário A — contrato teve pagamento
        (F.col("data_pagamento_aux").isNotNull()) &
        (F.col("data_vencimento") > F.col("data_pagamento_aux")),
        F.abs(
            F.datediff(
                F.col("data_vencimento"),
                F.col("data_pagamento_aux")
            )
        )
    ).otherwise(F.lit(0))
)

#Flag inadimplência

In [24]:
from pyspark.sql.functions import col, when

parcelas_main = parcelas_main.withColumn(
    "flag_inadimplencia_30_days",
    when(col("qtd_dias_de_atraso_v2")>30,1).otherwise(0)).withColumn(
     "flag_inadimplencia_90_days",when(col("qtd_dias_de_atraso_v2")>90,1).otherwise(0))

#parcelas_main.show(5, truncate=False)

In [28]:
#Validando flag-inadimplencia over todas parcelas
parcelas_main.select(
    "id_contrato",
    "parcela",
    "data_vencimento",
    "data_pagamento",
    "data_pagamento_aux",
    "flag_contrato_sem_pagamento",
    "qtd_dias_de_atraso_v2",
    "flag_inadimplencia_30_days",
    "flag_inadimplencia_90_days"
).orderBy("data_vencimento").filter(F.col("id_contrato") == "{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}").show(100, truncate=False)
#{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}
#{00012C6D-DD36-401A-8B26-F0C46C121114}

+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+---------------------+--------------------------+--------------------------+
|id_contrato                           |parcela|data_vencimento|data_pagamento|data_pagamento_aux|flag_contrato_sem_pagamento|qtd_dias_de_atraso_v2|flag_inadimplencia_30_days|flag_inadimplencia_90_days|
+--------------------------------------+-------+---------------+--------------+------------------+---------------------------+---------------------+--------------------------+--------------------------+
|{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}|1      |2024-11-27     |2024-11-17    |2024-11-17        |0                          |10                   |0                         |0                         |
|{B00ADF21-0E77-4C6B-9C10-A8A51B8C04BC}|2      |2024-12-27     |NULL          |2024-11-17        |0                          |40                   |1                         |0            